# 🎓 Tutor Virtual Inteligente — Notebook de Desarrollo

**Autora:** Yessica Peñaloza

Este notebook muestra, paso a paso, cómo construimos el cerebro del tutor:

1. Carga de datos
2. Exploración de datos
3. Vectorización TF-IDF
4. Cálculo de la similitud coseno
5. Pruebas del tutor virtual

La idea es entender QUÉ hace cada parte antes de empaquetarla en `utils/funciones.py`.

## 0. Preparación del entorno

Como el notebook vive dentro de `notebooks/`, agregamos la carpeta del proyecto
al `path` de Python para poder importar nuestro módulo `utils`.

In [ ]:
import os
import sys

# Subimos un nivel (de notebooks/ a la raíz del proyecto) y lo añadimos al path.
RUTA_PROYECTO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if RUTA_PROYECTO not in sys.path:
    sys.path.append(RUTA_PROYECTO)

# Librerías que usaremos.
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ruta a la base de conocimientos.
RUTA_CSV = os.path.join(RUTA_PROYECTO, "data", "conocimientos.csv")
print("CSV:", RUTA_CSV)

## 1. Carga de datos

Leemos el CSV con Pandas. El resultado es un `DataFrame`: una tabla con filas y columnas.

In [ ]:
from utils.funciones import cargar_base_conocimientos

# cargar_base_conocimientos lee el CSV y, ademas, separa las "varias formas
# de preguntar lo mismo" (variantes unidas por '|') en filas independientes.
df = cargar_base_conocimientos(RUTA_CSV)
df.head()

## 2. Exploración de datos

Veamos cuántas preguntas tenemos y cómo se reparten por categoría.

In [ ]:
print("Total de preguntas:", len(df))
print("\nColumnas:", list(df.columns))
print("\nPreguntas por categoría:")
df["categoria"].value_counts()

## 3. Vectorización TF-IDF

TF-IDF convierte cada pregunta en un vector de números que representa la
importancia de sus palabras. Entrenamos (`fit_transform`) el vectorizador con
todas las preguntas de la base.

In [ ]:
# Creamos el vectorizador (1 y 2 palabras como unidades).
vectorizador = TfidfVectorizer(ngram_range=(1, 2))

# Entrenamos con las preguntas y obtenemos la matriz TF-IDF.
matriz_tfidf = vectorizador.fit_transform(df["pregunta"])

print("Forma de la matriz (preguntas x términos):", matriz_tfidf.shape)
print("Primeros términos del vocabulario:", vectorizador.get_feature_names_out()[:15])

## 4. Cálculo de la similitud coseno

Tomamos una pregunta de prueba, la transformamos con el MISMO vectorizador y la
comparamos contra todas las preguntas de la base.

In [ ]:
pregunta_prueba = "¿Para qué sirve un bucle for?"

# Transformamos la pregunta (¡transform, no fit_transform!).
vector_prueba = vectorizador.transform([pregunta_prueba])

# Similitud contra todas las preguntas de la base.
similitudes = cosine_similarity(vector_prueba, matriz_tfidf)[0]

# Índice de la pregunta más parecida.
indice = similitudes.argmax()

print("Pregunta del usuario:", pregunta_prueba)
print("Pregunta más parecida:", df.iloc[indice]["pregunta"])
print("Similitud:", round(float(similitudes[indice]), 3))
print("\nRespuesta:\n", df.iloc[indice]["respuesta"])

## 5. Pruebas del tutor virtual

Ahora usamos la clase ya empaquetada en `utils/funciones.py`, que incluye la
limpieza del texto en español y el manejo de un umbral mínimo de similitud.

In [ ]:
from utils.funciones import TutorVirtual

tutor = TutorVirtual(RUTA_CSV)

preguntas = [
    "¿Qué es una variable?",
    "explicame la media",
    "que es tf idf",
    "¿Cómo cocinar una pizza?",  # fuera de tema: debe avisar que no sabe.
]

for p in preguntas:
    r = tutor.responder(p)
    print("=" * 70)
    print("Pregunta :", p)
    print("Respuesta:", r["respuesta"])
    print("Similitud:", r["similitud"], "| Categoría:", r["categoria"])

## 6. Tolerancia a errores de escritura

Gracias al vectorizador por n-gramas de caracteres (char_wb), el tutor
reconoce preguntas mal escritas aunque el error NO este en la base.
Comparamos la forma de las palabras, no solo las palabras exactas.

In [ ]:
# Typos que NO estan en el CSV: igual deben acertar gracias a los n-gramas.
typos = [
    'que es la programcion orientada a objetos',
    'que es un algorimo',
    'que es la regresion linal',
    'que es una varaible',
]

for t in typos:
    r = tutor.responder(t)
    print(f"[similitud {r['similitud']:.2f}] {t}")
    print('   ->', r['respuesta'][:70], '...')


## Conclusión

Con TF-IDF + similitud coseno logramos un tutor que entiende preguntas escritas
de distintas formas y devuelve la respuesta más relevante. Para mejorarlo se
podría: ampliar la base de conocimientos, usar embeddings semánticos o conectar
un modelo de lenguaje (LLM) para redactar respuestas más naturales.